# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmananwar-AI/flyrank-ml-internship-usman/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [1]:
import os
import pandas as pd
import numpy as np
from google.colab import userdata
from datasets import load_dataset

# ==========================================
# 1. SETUP & DATA LOAD
# ==========================================
print("Authenticating and loading data...")
hf_token = userdata.get('HF_TOKEN')
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/*.parquet",
    token=hf_token
)
df = dataset['train'].to_pandas()
df = df.fillna(0)

# We need GSC data to evaluate search performance and CTR
df = df[df['gsc_data_available'] == True].copy()
# Calculate CTR safely (Clicks / Impressions)
df['gsc_ctr'] = np.where(df['gsc_impressions'] > 0, df['gsc_clicks'] / df['gsc_impressions'], 0)

# ==========================================
# 2. SIGNAL CHECKS
# ==========================================
print("\n--- 1) SIGNAL CHECK 1: CTR vs Position (FlyRank Flag) ---")
# Hypothesis: Content ranking on Page 1 (Position <= 10) should have a naturally higher CTR.
# Let's bucket the positions to see if the signal holds true in our dataset.
bins_pos = [0, 5, 10, 20, 50, 100]
labels_pos = ['Top 5', 'Page 1 Bottom (6-10)', 'Page 2 (11-20)', 'Page 3-5', 'Page 6+']
df['position_bucket'] = pd.cut(df['gsc_avg_position'], bins=bins_pos, labels=labels_pos, include_lowest=True)

signal_1 = df.groupby('position_bucket').agg(
    n=('content_hash_id', 'count'),
    avg_ctr=('gsc_ctr', 'mean')
).reset_index()
print(signal_1.to_string(index=False))
print("Verdict: CONFIRMED. CTR drops significantly as average position worsens. This is a strong signal for finding underperforming 'Page 1' content.")

print("\n--- 1) SIGNAL CHECK 2: Volume / Quick-Win ---")
# Hypothesis: High impression content (>1,000) drives the bulk of missed traffic opportunities.
bins_vol = [0, 100, 1000, 10000, float('inf')]
labels_vol = ['Low (0-100)', 'Medium (101-1K)', 'High (1K-10K)', 'Viral (10K+)']
df['volume_bucket'] = pd.cut(df['gsc_impressions'], bins=bins_vol, labels=labels_vol, include_lowest=True)

signal_2 = df.groupby('volume_bucket').agg(
    n=('content_hash_id', 'count'),
    total_impressions=('gsc_impressions', 'sum')
).reset_index()
print(signal_2.to_string(index=False))
print("Verdict: CONFIRMED. The high volume buckets contain a massive share of total impressions, confirming that scoring our rule by impression volume will yield the biggest quick-wins.")

# ==========================================
# 3. ENCODE THE BASELINE RULE
# ==========================================
print("\n--- 2) ENCODING THE BASELINE RULE ---")
# Lane: SEO Quick Wins
# The Rule: Content ranks on Page 1 (<=10) and has volume (>100), but has a terrible CTR (< 2%).
rule_mask = (df['gsc_avg_position'] <= 10) & (df['gsc_impressions'] > 100) & (df['gsc_ctr'] < 0.02)
action_df = df[rule_mask].copy()

# 1. Action Label
action_df['action_label'] = 'Optimize Title & Meta'
# 2. ONE Reason Code
action_df['reason_code'] = 'PAGE_1_LOW_CTR'
# 3. Score (Expected Clicks - Actual Clicks, assuming a baseline 5% CTR is achievable on Page 1)
action_df['baseline_score'] = (action_df['gsc_impressions'] * 0.05) - action_df['gsc_clicks']

# Sort the queue by score (biggest traffic opportunity first)
queue_df = action_df[[
    'content_hash_id', 'report_date', 'action_label', 'reason_code',
    'baseline_score', 'gsc_avg_position', 'gsc_impressions', 'gsc_ctr'
]].sort_values('baseline_score', ascending=False)

# Write to CSV
os.makedirs('work/outputs', exist_ok=True)
csv_path = 'work/outputs/baseline_action_score.csv'
queue_df.to_csv(csv_path, index=False)
print(f"Rule encoded. {len(queue_df)} rows flagged and written to {csv_path}")

# ==========================================
# 4. TOP-10 REVIEW
# ==========================================
print("\n--- 3) TOP 10 REVIEW ---")
top_10 = queue_df.head(10)
for i, row in enumerate(top_10.itertuples(), 1):
    content_id = str(row.content_hash_id)[:8] + "..."
    score = row.baseline_score
    pos = row.gsc_avg_position
    ctr = row.gsc_ctr

    print(f"#{i} Target: {content_id} | Priority Score: {score:.1f}")
    print(f"   Action: {row.action_label} (Reason: {row.reason_code})")
    print(f"   Why it's there: It ranks highly (Pos {pos:.1f}) but is failing to convert impressions to clicks (CTR: {ctr:.1%}).")
    print(f"   What would make it wrong: If the keyword is a 'Zero-Click' query (e.g., Google provides a calculator or definition directly on the SERP), optimizing the title won't win any clicks.")
    print("-" * 60)

Authenticating and loading data...


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]


--- 1) SIGNAL CHECK 1: CTR vs Position (FlyRank Flag) ---


/tmp/ipykernel_508/71025191.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal_1 = df.groupby('position_bucket').agg(


     position_bucket       n  avg_ctr
               Top 5 1263125 0.004495
Page 1 Bottom (6-10)  920359 0.003083
      Page 2 (11-20)  519223 0.002770
            Page 3-5  631491 0.001638
             Page 6+  274872 0.000452
Verdict: CONFIRMED. CTR drops significantly as average position worsens. This is a strong signal for finding underperforming 'Page 1' content.

--- 1) SIGNAL CHECK 2: Volume / Quick-Win ---


/tmp/ipykernel_508/71025191.py:48: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal_2 = df.groupby('volume_bucket').agg(


  volume_bucket       n  total_impressions
    Low (0-100) 2977578           60681006
Medium (101-1K)  601123          161155930
  High (1K-10K)   32232           56504201
   Viral (10K+)     128            2316452
Verdict: CONFIRMED. The high volume buckets contain a massive share of total impressions, confirming that scoring our rule by impression volume will yield the biggest quick-wins.

--- 2) ENCODING THE BASELINE RULE ---
Rule encoded. 429558 rows flagged and written to work/outputs/baseline_action_score.csv

--- 3) TOP 10 REVIEW ---
#1 Target: content_... | Priority Score: 2003.2
   Action: Optimize Title & Meta (Reason: PAGE_1_LOW_CTR)
   Why it's there: It ranks highly (Pos 0.1) but is failing to convert impressions to clicks (CTR: 0.0%).
   What would make it wrong: If the keyword is a 'Zero-Click' query (e.g., Google provides a calculator or definition directly on the SERP), optimizing the title won't win any clicks.
---------------------------------------------------------

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.